<a href="https://colab.research.google.com/github/ImvaibT/AiResidncyDDSdubai-Cohrt10-20260411/blob/DDS_all_MCs-Codesnpts/Modfy-hugngfc_CodesPy./Copy_of_Copy_of_Day_2_rag_llamaindex_mar26_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama_index llama-index-readers-file

In [ ]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('Openaiky')

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

9
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Data

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()
response = query_engine.query("What is standard working hours in Decoding Data Science")
print(response)

Standard working hours in Decoding Data Science are from 9:00 to 18:00, Monday to Friday.


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-5.4-nano", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''

You are Ayesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Your objective is to interact politely and professionally with employees, answering only HR-related questions. Use only information directly from the 4 connected HR documents to provide your answers. Always provide an explicit citation indicating the document source for every answer. Do not offer information or suggestions beyond what is present in these documents.

If the requested information cannot be found in the connected documents, politely instruct the user to email connect@decodingdatascience.com for further assistance. For questions outside of HR  inform the user that you can only answer HR-related questions.
'''

#documents = SimpleDirectoryReader("data").load_data()

index = VectorStoreIndex.from_documents(documents=documents)

# Configure query engine with system prompt
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

DDS standard office hours in Dubai are **9:00–18:00, Monday–Friday**.


In [ ]:
import time
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Start timer
start_time = time.time()

# Load and index documents
index = VectorStoreIndex.from_documents(documents=documents)

# Query the index
query_engine = index.as_query_engine()
response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

# End timer and print duration
end_time = time.time()
print(f"\nExecution Time: {end_time - start_time:.2f} seconds")


DDS’s standard office hours in Dubai are **9:00–18:00, Monday–Friday**.

Execution Time: 1.08 seconds


In [ ]:
import gradio as gr
import openai
from google.colab import userdata
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.llms import ChatMessage # Import ChatMessage

# --- 1. INITIALIZATION & SETUP ---
# Retrieve API key (with a fallback just in case you move this script out of Colab later)
try:
    openai.api_key = userdata.get('Openaiky')
except Exception:
    import os
    openai.api_key = os.environ.get("OPENAI_API_KEY")

# Define Ayesha's Persona
system_prompt_str = (
    "You are Ayesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. "
    "Your objective is to interact politely and professionally with employees, answering only HR-related questions. "
    "Use only information directly from the connected HR documents to provide your answers. "
    "Always provide an explicit citation indicating the document source for every answer. "
    "Do not offer information or suggestions beyond what is present in these documents. "
    "If the requested information cannot be found, politely instruct the user to email connect@decodingdatascience.com. "
    "For questions outside of HR, inform the user that you can only answer HR-related questions."
)

# Configure LLM and Embeddings exactly as you defined
# Pass the system_prompt_str directly to the OpenAI LLM
Settings.llm = OpenAI(model="gpt-5.4-nano", temperature=0.2, system_prompt=system_prompt_str)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# The ChatMessage conversion is no longer needed here as system_prompt is passed directly to LLM
# system_prompt = ChatMessage(role="system", content=system_prompt_str)

# Load data and create index (Assuming 'data' folder is populated)
print("Initializing Knowledge Base...")
documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"]
).load_data()

index = VectorStoreIndex.from_documents(documents=documents)

# Upgrade to a Chat Engine so it remembers conversation history
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    # Remove prefix_messages as it's not supported by CondenseQuestionChatEngine
    # The system prompt is now handled by the LLM configuration in Settings.llm
    verbose=True
)

# --- 2. BACKEND CHAT FUNCTION ---
def chat_with_ayesha(user_message, history):
    """
    This function takes the user input and the Gradio chat history,
    passes it to LlamaIndex, and yields the response.
    """
    if not user_message.strip():
        return "", history

    # Append user message to history immediately for a snappy UI feel
    history.append((user_message, None))
    yield "", history

    # Query the LlamaIndex chat engine
    response = chat_engine.chat(user_message)

    # Update the history with the AI's response
    history[-1] = (user_message, response.response)
    yield "", history

def clear_chat():
    """Resets the LlamaIndex chat memory and clears the UI."""
    chat_engine.reset()
    return []

# --- 3. FRONTEND UI ARCHITECTURE ---
# Using Soft theme for a modern, clean look
with gr.Blocks(theme=gr.themes.Soft(), fill_width=True) as app:

    # Header Section
    with gr.Row():
        gr.Markdown(
            """
            # 🏢 Decoding Data Science HR Portal
            ### *Ask Ayesha: Your Enterprise HR Assistant*
            """
        )

    # Main Dashboard Layout
    with gr.Row():

        # Left Sidebar: Information & Branding (Traditional UI split)
        with gr.Column(scale=1, min_width=300):
            gr.Image(value="https://cdn-icons-png.flaticon.com/512/4712/4712010.png", width=150, show_label=False, show_download_button=False)
            gr.Markdown("### 🤖 Agent Status: **Online**")
            gr.Markdown("**Role:** Enterprise HR Specialist")
            gr.Markdown("**Knowledge Base:** Connected to 4 DDS internal HR PDFs.")

            with gr.Accordion("How to use this tool", open=False):
                gr.Markdown(
                    """
                    - Ask about leave policies, office hours, or benefits.
                    - Ayesha will cite the specific document she pulled the answer from.
                    - For non-HR issues, please contact IT.
                    """
                )

            clear_btn = gr.Button("🗑️ Clear Conversation", variant="secondary")

        # Right Column: Heavy UI Interaction (The Chat Interface)
        with gr.Column(scale=3):
            # The chatbot component holds the visual history
            chatbot = gr.Chatbot(
                label="Conversation with Ayesha",
                height=500,
                bubble_full_width=False,
                avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/4712/4712139.png") # User vs AI avatars
            )

            # Input Area
            with gr.Row():
                msg_input = gr.Textbox(
                    show_label=False,
                    placeholder="Type your HR question here and press Enter...",
                    container=False,
                    scale=4
                )
                submit_btn = gr.Button("Send ✉️", variant="primary", scale=1)

    # --- 4. EVENT BINDING ---
    # Trigger chat when hitting 'Enter' in textbox
    msg_input.submit(
        fn=chat_with_ayesha,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

    # Trigger chat when clicking the 'Send' button
    submit_btn.click(
        fn=chat_with_ayesha,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

    # Trigger clear memory and UI
    clear_btn.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot]
    )

# Launch the app
app.launch(debug=True)

Initializing Knowledge Base...


/tmp/ipykernel_696/745054117.py:82: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), fill_width=True) as app:
/tmp/ipykernel_696/745054117.py:117: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_696/745054117.py:117: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_696/745054117.py:117: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set a

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b5af7458a0f9efefe9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Querying with: leaves policies in dds
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2c4ebd0dd1606c4b65.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://b5af7458a0f9efefe9.gradio.live


In [ ]:
import gradio as gr
print(gr.__version__)


5.50.0



sdk_version: 6.16.0

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Load data and build the index
#documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()

# Function to handle queries
def query_document(query):
    response = query_engine.query(query)
    return str(response)

# Gradio interface
interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="DDS Enterise Chatbot connect to Data",
    description="Ask questions about the documents loaded into the system."
)

# Launch the Gradio app
if __name__ == "__main__":
    interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c4ebd0dd1606c4b65.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
print(gr.__version__)


5.50.0


In [ ]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    #documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Either way we can now query the index
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)


The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [ ]:
#with the timer
import os
import time
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# Start timer for index setup
start_time = time.time()

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

setup_duration = time.time() - start_time
print(f"Index setup time: {setup_duration:.2f} seconds")

# Start timer for query
query_start_time = time.time()

# Prepare the query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
query_engine = RetrieverQueryEngine(retriever=retriever)

# Execute query
response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

query_duration = time.time() - query_start_time
print(f"Query time: {query_duration:.2f} seconds")


Index setup time: 0.78 seconds
The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.
Query time: 1.03 seconds


In [ ]:
#running without creating Storage
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

import time
start_time = time.time()
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai??")
print(response)


end_time = time.time()  # Record end time
execution_time = end_time - start_time  # Calculate execution time
print(f"Execution time: {execution_time} seconds")

The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.
Execution time: 0.990729808807373 seconds
